In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout

from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import Callback

## Mounting to Google Drive

### Uncomment if utilizing Google Colab

In [2]:
colab = False

In [3]:
path = os.getcwd()
print(path)
if colab:
    from google.colab import drive 
    drive.mount('/content/drive')
    os.chdir("./drive/My Drive/TreatmentSystems/Final_Dataset")
else:
    os.chdir(os.path.join(path, 'Final_Dataset'))

print(os.getcwd())

c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\Final_Dataset


### Receiving Data and Completing Train-Test Split

In [4]:
data = pd.read_csv('concatenated_data.csv')
print(type(data))
print(data.head())

# is_sorted = data.equals(data.sort_values(by=['PatientID', 'time']).reset_index(drop=True))
# print("Data is sorted correctly:", is_sorted)
times = data.pop('time')

data.head()

numPatients = data['PatientID'].max()
print(numPatients)

<class 'pandas.core.frame.DataFrame'>
               time  PatientID  calories  heart_rate  steps  basal_rate  \
0  06-13-2018 18:40          1    6.3595   82.322835     34    0.091667   
1  06-13-2018 18:45          1    7.7280   83.740157      0    0.091667   
2  06-13-2018 18:50          1    4.7495   80.525180      0    0.091667   
3  06-13-2018 18:55          1    6.3595   89.129032     20    0.091667   
4  06-13-2018 19:00          1    5.1520   92.495652      0    0.075000   

   bolus_volume_delivered  glucose  
0                     0.0    332.0  
1                     0.0    326.0  
2                     0.0    330.0  
3                     0.0    324.0  
4                     0.0    306.0  
22


In [5]:
print((data['basal_rate'] >= 0).all() and (data['basal_rate'] <= 1).all())

True


In [6]:
print((data['bolus_volume_delivered'] >= 0).all() and (data['bolus_volume_delivered'] <= 1).all())

False


In [7]:
train_patients = list(range(1, 17))
print(len(train_patients))
print(train_patients)

test_patients = list(range(17, numPatients + 1))
print(len(test_patients))
print(test_patients)

train_data = data[data['PatientID'].isin(train_patients)]
test_data = data[data['PatientID'].isin(test_patients)]

print("\n Training Data:")
train_X, train_y = train_data.loc[:, data.columns != 'glucose'], train_data['glucose']
print(train_X.head())
print(train_y.head())

train_len = len(train_X)
N = len(data)
print(train_len, N)
train_percent = (train_len / N)

print("Percent of Data in Training = ", np.round(train_percent * 100, 2))
print("\n Testing Data:")
test_X, test_y = test_data.loc[:, data.columns != 'glucose'], test_data['glucose']
print(test_X.head())
print(test_y.head())

print("Percent of Data in Testing = ", np.round((1 - train_percent) * 100, 2))

16
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
6
[17, 18, 19, 20, 21, 22]

 Training Data:
   PatientID  calories  heart_rate  steps  basal_rate  bolus_volume_delivered
0          1    6.3595   82.322835     34    0.091667                     0.0
1          1    7.7280   83.740157      0    0.091667                     0.0
2          1    4.7495   80.525180      0    0.091667                     0.0
3          1    6.3595   89.129032     20    0.091667                     0.0
4          1    5.1520   92.495652      0    0.075000                     0.0
0    332.0
1    326.0
2    330.0
3    324.0
4    306.0
Name: glucose, dtype: float64
46191 63498
Percent of Data in Training =  72.74

 Testing Data:
       PatientID  calories  heart_rate  steps  basal_rate  \
46191         17   15.0429   96.371901      8       0.035   
46192         17    8.3164   91.395349      0       0.035   
46193         17    7.5826   85.991935      0       0.035   
46194         17    7.3380   82.434

### Scale the Features Before Training

#### Columns That Needed to Be Scaled:
- Calories
- Steps
- Heart Rate
- Glucose

Possibly Need to Scale the Basal Rate -> First Attempt will be without scaling the Basal

Note that Glucose Needs to be converted back to the original values before the scaling in order to interpret the results

In [36]:
print("Calories:")
print("Min = ", train_X['calories'].min())
print("Max = ", train_X['calories'].max())

print("Heart Rate:")
print("Min = ", train_X['heart_rate'].min())
print("Max = ", train_X['heart_rate'].max())

print("Steps:")
print("Min = ", train_X['steps'].min())
print("Max = ", train_X['steps'].max())

print("Basal Rate:")
print("Min = ", train_X['basal_rate'].min())
print("Max = ", train_X['basal_rate'].max())

Calories:
Min =  4.025000036
Max =  78.85771084
Heart Rate:
Min =  39.38406616
Max =  174.7142857
Steps:
Min =  0
Max =  658
Basal Rate:
Min =  0.0
Max =  0.25


In [8]:
scaler1_x = MinMaxScaler(feature_range=(-1, 1))
scaler1_y = MinMaxScaler(feature_range=(-1, 1))

In [46]:
column_names = data.columns
print(column_names[-3:-1])
print(column_names[:1])
print(column_names[1:data.shape[1] - 3])
standardize_features = list(column_names[1:data.shape[1] - 3]) # Features to standardize
passthrough_features = list(column_names[:1]) + list(column_names[-3:-1]) # Keep these unchanged
print(standardize_features, passthrough_features)

# Define ColumnTransformer
scaler1_x = ColumnTransformer(
    transformers=[
        ('passthrough', 'passthrough', passthrough_features), 
        ('standardize', MinMaxScaler(), standardize_features)
    ]
)

train_X_scaled = scaler1_x.fit_transform(train_X)
test_X_scaled = scaler1_x.transform(test_X)

print(train_X_scaled[:5, 1:4], type(train_X_scaled))
print(test_X_scaled[:5, 1:4], type(test_X_scaled))

Index(['basal_rate', 'bolus_volume_delivered'], dtype='object')
Index(['PatientID'], dtype='object')
Index(['calories', 'heart_rate', 'steps'], dtype='object')
['calories', 'heart_rate', 'steps'] ['PatientID', 'basal_rate', 'bolus_volume_delivered']
[[0.09166667 0.         0.03119625]
 [0.09166667 0.         0.04948371]
 [0.09166667 0.         0.0096816 ]
 [0.09166667 0.         0.03119625]
 [0.075      0.         0.01506026]] <class 'numpy.ndarray'>
[[0.035      0.         0.14723374]
 [0.035      0.         0.05734658]
 [0.035      0.         0.04754071]
 [0.035      0.         0.04427208]
 [0.035      0.         0.04754071]] <class 'numpy.ndarray'>


In [38]:
print("Glucose")
print("Min = ", train_y.min())
print("Max = ", train_y.max())

Glucose
Min =  40.0
Max =  444.0


In [41]:
scaler1_y = MinMaxScaler()
train_y_scaled = scaler1_y.fit_transform(np.array(train_y).reshape(-1, 1))
test_y_scaled = scaler1_y.transform(np.array(test_y).reshape(-1, 1))

print(train_y_scaled[:5])
print(test_y_scaled[:5])

[[0.72277228]
 [0.70792079]
 [0.71782178]
 [0.7029703 ]
 [0.65841584]]
[[0.        ]
 [0.00330033]
 [0.00660066]
 [0.00990099]
 [0.02475248]]


In [ ]:
# scaler2 = MinMaxScaler(feature_range=(0, 1))
# data = scaler2.fit_transform(data)

### Creation of Plots for Visualizations